# 01 · Modelo base DERM12345 — benigno vs. sospechoso de melanoma

**Fase A del plan.** Primer modelo de referencia: todo lo que se pruebe después
(BCN20000, preentreno con HAM10000, más resolución…) se compara contra estos números.

- **Datos:** 896 imágenes acrales de DERM12345 (760 nevus · 136 melanomas ALM/ANM).
- **Split:** el test del paper (sin mezcla de pacientes, verificado). La validación sale
  del train **agrupando por `patient_id`**. El test **no se toca** hasta el final.
- **Modelo:** EfficientNet-B0 preentrenada en ImageNet (torchvision), salida binaria.
- **Métrica principal:** sensibilidad para melanoma a un umbral elegido en validación
  (es un triaje: un falso negativo es lo más caro), más AUC.

### Cómo ejecutarlo en Colab
1. Abre el notebook en Colab:
   <https://colab.research.google.com/github/Mangelits/AcralisisPodoScan/blob/main/notebooks/01_baseline_derm12345.ipynb>
2. *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.
3. *Entorno de ejecución → Ejecutar todas*. Pedirá permiso para montar Google Drive
   (ahí se guardan las imágenes, para no re-descargarlas, y el modelo entrenado).

In [ ]:
# ---- Configuración -------------------------------------------------------
REPO_URL = "https://github.com/Mangelits/AcralisisPodoScan.git"
BRANCH = "main"
USE_DRIVE = True          # guarda imágenes y modelo en Google Drive

SEED = 42
IMG_SIZE = 224            # resolución nativa de EfficientNet-B0
CACHE_SIZE = 384          # lado corto de las imágenes pre-redimensionadas
EPOCHS = 25
BATCH_SIZE = 32
LR = 3e-4
WEIGHT_DECAY = 1e-4
VAL_FOLDS = 6             # ~1/6 de los pacientes de train van a validación
TARGET_SENSITIVITY = 0.90 # el umbral se elige para detectar ≥90 % de melanomas en val

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/AcralisisPodoScan").exists():
        !git clone -q -b {BRANCH} {REPO_URL} /content/AcralisisPodoScan
    %cd /content/AcralisisPodoScan
    !git pull -q
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
if IN_COLAB and USE_DRIVE:
    STORE = Path("/content/drive/MyDrive/acralis")
    RAW_DIR, MODELS_DIR = STORE / "data" / "raw" / "derm12345", STORE / "models"
else:  # en local, todo dentro de data/ (ignorado por git)
    RAW_DIR, MODELS_DIR = ROOT / "data" / "raw" / "derm12345", ROOT / "data" / "processed" / "models"
CACHE_DIR = ROOT / "data" / "processed" / f"derm12345_{CACHE_SIZE}"
for d in (RAW_DIR, CACHE_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "\nRAW_DIR:", RAW_DIR, "\nMODELS_DIR:", MODELS_DIR)

## 1 · Datos

Descarga solo las 896 imágenes acrales (reanudable: si ya están en Drive, no baja nada).
Después se pre-redimensionan una vez a lado corto 384 px: las originales son de hasta
4032×3024 y decodificarlas en cada época haría el entrenamiento muy lento.

In [ ]:
!python scripts/datasets/download_derm12345.py --out "{RAW_DIR}"

# Si la descarga falla, parar aquí (si no, todo lo siguiente sale vacío).
acral_csv = RAW_DIR / "acral.csv"
n_rows = sum(1 for _ in open(acral_csv)) - 1 if acral_csv.exists() else 0
assert n_rows == 896, (
    f"acral.csv tiene {n_rows} de 896 imágenes. Revisa los errores (✗) de la salida "
    "de arriba y vuelve a ejecutar esta celda: la descarga es reanudable."
)
print("✓ 896 imágenes acrales listas")

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageOps

def cache_image(src: Path) -> None:
    dst = CACHE_DIR / src.name
    if dst.exists():
        return
    img = ImageOps.exif_transpose(Image.open(src)).convert("RGB")
    scale = CACHE_SIZE / min(img.size)
    if scale < 1:
        img = img.resize((round(img.width * scale), round(img.height * scale)), Image.LANCZOS)
    img.save(dst, quality=95)

sources = sorted((RAW_DIR / "images").glob("*.jpg"))
with ThreadPoolExecutor(8) as pool:
    list(pool.map(cache_image, sources))
print(len(sources), "imágenes en caché en", CACHE_DIR)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

df = pd.read_csv(RAW_DIR / "acral.csv")
df["target"] = (df["malignancy"] == "malignant").astype(int)   # 1 = sospechoso de melanoma
df["path"] = [str(CACHE_DIR / Path(f).name) for f in df["file"]]

test_df = df[df["split"] == "test"].reset_index(drop=True)
trainval = df[df["split"] == "train"].reset_index(drop=True)

# Validación agrupada por paciente y estratificada por clase.
sgkf = StratifiedGroupKFold(n_splits=VAL_FOLDS, shuffle=True, random_state=SEED)
tr_idx, va_idx = next(sgkf.split(trainval, trainval["target"], groups=trainval["patient_id"]))
train_df, val_df = trainval.iloc[tr_idx].reset_index(drop=True), trainval.iloc[va_idx].reset_index(drop=True)

# Ningún paciente puede estar en dos particiones.
parts = {"train": train_df, "val": val_df, "test": test_df}
for a in parts:
    for b in parts:
        if a < b:
            assert not set(parts[a]["patient_id"]) & set(parts[b]["patient_id"]), (a, b)

summary = pd.DataFrame({
    k: {"benigno": int((v.target == 0).sum()), "melanoma": int((v.target == 1).sum()),
        "pacientes": v.patient_id.nunique()}
    for k, v in parts.items()
}).T
summary

## 2 · Dataset y aumento de datos

Las lesiones dermatoscópicas no tienen orientación canónica → volteos y rotaciones libres.
El color es diagnóstico (p. ej. patrón paralelo de la cresta), así que el *jitter* de color es suave.

In [ ]:
import random
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
if DEVICE.type != "cuda":
    print("⚠️ Sin GPU: activa una en Entorno de ejecución → Cambiar tipo de entorno.")

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)   # ImageNet
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(180),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([
    T.Resize(IMG_SIZE), T.CenterCrop(IMG_SIZE), T.ToTensor(), T.Normalize(MEAN, STD),
])

class LesionDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tf):
        self.paths, self.targets, self.tf = frame["path"].tolist(), frame["target"].tolist(), tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), torch.tensor(self.targets[i], dtype=torch.float32)

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=DEVICE.type == "cuda")
train_dl = DataLoader(LesionDataset(train_df, train_tf), shuffle=True, drop_last=True, **loader_kw)
val_dl = DataLoader(LesionDataset(val_df, eval_tf), **loader_kw)
test_dl = DataLoader(LesionDataset(test_df, eval_tf), **loader_kw)

## 3 · Modelo y entrenamiento

- EfficientNet-B0 con pesos de ImageNet; se sustituye el clasificador por una salida (logit).
- Desbalance ~1:6 → `pos_weight` en la pérdida para que fallar un melanoma cueste más.
- Se guarda la época con **mejor AUC en validación**.

In [ ]:
import copy
import time
from sklearn.metrics import roc_auc_score
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, 1)
model = model.to(DEVICE)

pos_weight = torch.tensor((train_df.target == 0).sum() / (train_df.target == 1).sum(), device=DEVICE)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, epochs=EPOCHS,
                                                steps_per_epoch=len(train_dl), pct_start=0.15)
scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")

@torch.no_grad()
def predict(dl) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    probs, ys = [], []
    for x, y in dl:
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            logits = model(x.to(DEVICE, non_blocking=True)).squeeze(1)
        probs.append(torch.sigmoid(logits.float()).cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(probs)

history, best_auc, best_state = [], -1.0, None
for epoch in range(1, EPOCHS + 1):
    model.train(); t0, running = time.time(), 0.0
    for x, y in train_dl:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            loss = criterion(model(x).squeeze(1), y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item() * len(y)
    y_val, p_val = predict(val_dl)
    auc = roc_auc_score(y_val, p_val)
    history.append({"epoch": epoch, "loss": running / len(train_dl.dataset), "val_auc": auc})
    if auc > best_auc:
        best_auc, best_epoch, best_state = auc, epoch, copy.deepcopy(model.state_dict())
    print(f"época {epoch:2d} · loss {history[-1]['loss']:.4f} · val AUC {auc:.3f}"
          f"{'  ← mejor' if epoch == best_epoch else ''} · {time.time() - t0:.0f}s")

model.load_state_dict(best_state)
print(f"\nMejor época: {best_epoch} (val AUC {best_auc:.3f})")

In [ ]:
import matplotlib.pyplot as plt

h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(h.epoch, h.loss); ax[0].set(title="Pérdida de entrenamiento", xlabel="época")
ax[1].plot(h.epoch, h.val_auc); ax[1].axvline(best_epoch, ls="--", c="grey")
ax[1].set(title="AUC en validación", xlabel="época", ylim=(0.5, 1))
plt.tight_layout(); plt.show()

## 4 · Umbral (en validación) y evaluación final (en test)

El umbral se fija **solo con validación**: el menor que detecta al menos el
`TARGET_SENSITIVITY` de los melanomas. Después se aplica tal cual al test.
Con solo 28 melanomas en test las métricas tienen mucha incertidumbre → se dan
intervalos de confianza al 95 % por *bootstrap*.

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve

y_val, p_val = predict(val_dl)
fpr, tpr, thr = roc_curve(y_val, p_val)
threshold = float(thr[np.argmax(tpr >= TARGET_SENSITIVITY)])
print(f"Umbral elegido en validación: {threshold:.3f}")

y_test, p_test = predict(test_dl)
pred = (p_test >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()

def bootstrap_ci(fn_metric, n=2000, rng=np.random.default_rng(SEED)):
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y_test), len(y_test))
        if y_test[i].min() != y_test[i].max():
            vals.append(fn_metric(y_test[i], p_test[i], pred[i]))
    return np.percentile(vals, [2.5, 97.5])

metrics = {
    "auc": roc_auc_score(y_test, p_test),
    "sensibilidad": tp / (tp + fn),
    "especificidad": tn / (tn + fp),
    "vpp": tp / (tp + fp) if tp + fp else float("nan"),
    "vpn": tn / (tn + fn) if tn + fn else float("nan"),
}
ci = {
    "auc": bootstrap_ci(lambda y, p, _: roc_auc_score(y, p)),
    "sensibilidad": bootstrap_ci(lambda y, _, q: q[y == 1].mean()),
    "especificidad": bootstrap_ci(lambda y, _, q: 1 - q[y == 0].mean()),
}
print(f"\nTEST · {len(y_test)} imágenes ({int(y_test.sum())} melanomas)")
for k, v in metrics.items():
    extra = f"  (IC95 {ci[k][0]:.2f}–{ci[k][1]:.2f})" if k in ci else ""
    print(f"  {k:<14}{v:.3f}{extra}")
print(f"\nMatriz de confusión  ·  melanomas detectados {tp}/{tp + fn}  ·  falsas alarmas {fp}/{fp + tn}")

In [ ]:
fpr_t, tpr_t, _ = roc_curve(y_test, p_test)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(fpr_t, tpr_t, label=f"AUC {metrics['auc']:.3f}")
ax[0].plot([0, 1], [0, 1], ls="--", c="grey")
ax[0].scatter(1 - metrics["especificidad"], metrics["sensibilidad"], c="red", zorder=3, label="umbral elegido")
ax[0].set(title="Curva ROC (test)", xlabel="1 − especificidad", ylabel="sensibilidad"); ax[0].legend()
cm = np.array([[tn, fp], [fn, tp]])
ax[1].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax[1].text(j, i, v, ha="center", va="center", fontsize=14)
ax[1].set(title="Matriz de confusión (test)", xticks=[0, 1], yticks=[0, 1],
          xticklabels=["pred. benigno", "pred. sospechoso"], yticklabels=["benigno", "melanoma"])
plt.tight_layout(); plt.show()

## 5 · Guardar el modelo y los resultados

Se guardan en `MyDrive/acralis/models/` (no en git: los `.pt` están en `.gitignore`).
Apunta las métricas del JSON en el documento de resultados: son la **línea base**.

In [ ]:
import json
from datetime import datetime

run_id = f"effb0_derm12345_{datetime.now():%Y%m%d_%H%M}"
torch.save({
    "state_dict": model.state_dict(),
    "arch": "efficientnet_b0", "img_size": IMG_SIZE, "mean": MEAN, "std": STD,
    "threshold": threshold, "classes": ["benigno", "sospechoso_melanoma"],
}, MODELS_DIR / f"{run_id}.pt")

report = {
    "run_id": run_id, "best_epoch": best_epoch, "val_auc": best_auc, "threshold": threshold,
    "test": {k: float(v) for k, v in metrics.items()},
    "test_ci95": {k: [float(a), float(b)] for k, (a, b) in ci.items()},
    "confusion": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    "splits": summary.to_dict(orient="index"),
    "config": {"seed": SEED, "img_size": IMG_SIZE, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
               "lr": LR, "weight_decay": WEIGHT_DECAY, "target_sensitivity": TARGET_SENSITIVITY},
    "history": history,
}
(MODELS_DIR / f"{run_id}.json").write_text(json.dumps(report, indent=2, ensure_ascii=False))
print("Guardado en", MODELS_DIR / run_id)